<a href="https://colab.research.google.com/github/antonypradeep54/Agent-Autogen-multi-agent-eda/blob/main/6_C13_M3_L10_Project_AutoGen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<font size=+3 color="#009688"><center><b> Project : AutoGen </b></center></font>

## Table of contents

- Problem statement
- Libraries and configurations
- Dedining agents
    - Data preparation agent
    - EDA agent
    - Report generator agent
    - Critic agent
    - Executor agent
    - Admin agent
    - User proxy agent
- Group chat orchestration
- EDA workflow initiation
- Groupchat output
- Data visualization
- References

##Problem Statement

Exploratory Data Analysis (EDA) is a fundamental step in data science that involves understanding the
structure, characteristics, and insights of a dataset before applying advanced modeling techniques.
However, conducting EDA is often a complex and iterative process, requiring diverse tasks such as
data cleaning, preprocessing, statistical summarization, and visualization. Managing these tasks
efficiently in a collaborative environment can be challenging, especially when multiple team
members or roles are involved. Ensuring the accuracy of the analysis, incorporating feedback, and
producing a high-quality report that meets the required standards adds additional complexity. </br> </br>
This project addresses these challenges by developing a multi-agent system to streamline and
automate the EDA process. Each agent in the system is assigned a specific role, such as preparing
data, conducting EDA, generating reports, or providing critical feedback. These roles ensure task
specialization and modularity, enabling a structured workflow. The process is managed within a
collaborative environment where agents communicate, coordinate tasks, and incorporate feedback
to iteratively improve the analysis. Additionally, an Executor agent verifies the accuracy of code and
outputs, while an Admin agent oversees the entire process to ensure compliance with project goals
and standards. </br></br>
The overarching goal of this system is to produce a comprehensive and well-structured EDA report.
This report should include a data overview, key insights, visualizations, and a summary of findings. It
must reflect feedback provided by a Critic agent to ensure clarity, accuracy, and actionable insights.
By automating and organizing the EDA process, this framework ensures efficiency, reproducibility,
and high-quality results in data exploration projects.

##Libraries and configurations

In [ ]:
%%capture
!pip install pyautogen
!pip install autogen
!pip install pyautogen==0.2.0

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OpenAI_API_Key')

In [ ]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os

# Basic LLM config
config_list = [
    {"model": "gpt-4o-mini", "api_key": os.environ["OPENAI_API_KEY"]}
]

##Data Preparation Agent

In [ ]:
data_preparation_agent = AssistantAgent(
    name="DataPreparer",
    system_message=(
        "You are a Data Preparation Agent. "
        "Your task is to load or generate datasets, clean missing values, remove duplicates, "
        "and save the cleaned dataset for further analysis."
    ),
    llm_config={"config_list": config_list}
)

##EDA Agent

In [ ]:
eda_agent = AssistantAgent(
    name="EDAAgent",
    system_message=(
        "You are an EDA Agent. Perform exploratory data analysis on a given dataset, "
        "compute descriptive statistics, correlations, and generate insights."
    ),
    llm_config={"config_list": config_list}
)

##Report Generator Agent

In [ ]:
report_generator_agent = AssistantAgent(
    name="ReportGenerator",
    system_message=(
        "You are a Report Generator Agent. Use outputs from the EDA Agent to create a clear, "
        "structured, and concise EDA report summarizing insights and visualizations."
    ),
    llm_config={"config_list": config_list}
)

##Critic Agent

In [ ]:
critic_agent = AssistantAgent(
    name="Critic",
    system_message=(
        "You are a Critic Agent. Review the EDA report for accuracy, completeness, "
        "and actionable insights. Provide constructive feedback to improve it."
    ),
    llm_config={"config_list": config_list}
)

##Executor Agent

In [ ]:
executor_agent = AssistantAgent(
    name="Executor",
    system_message=(
        "You are an Executor Agent. Your job is to verify that all code runs correctly, "
        "validate outputs, and confirm that each agent completed its task successfully."
    ),
    llm_config={"config_list": config_list}
)

##Admin Agent

In [ ]:
admin_agent = AssistantAgent(
    name="Admin",
    system_message=(
        "You are the Admin Agent. Oversee the workflow, coordinate agents, "
        "ensure tasks follow in the correct sequence, and that the project goals are met."
    ),
    llm_config={"config_list": config_list}
)

##User Proxy Agent

In [ ]:
user_proxy_agent = UserProxyAgent(
    name="UserProxy",
    human_input_mode="NEVER",
    llm_config={"config_list": config_list}
)

##Group Chat Orchestration

In [ ]:
groupchat = GroupChat(
    agents=[
        user_proxy_agent,
        data_preparation_agent,
        eda_agent,
        report_generator_agent,
        critic_agent,
        executor_agent,
        admin_agent
    ],
    messages=[],
    max_round=25,
    #terminate_on_keywords=["TERMINATE"]
)

manager = GroupChatManager(groupchat=groupchat, llm_config={"config_list": config_list})

##EDA Workflow Initiation

In [ ]:
%%capture
initial_task = (
    "Prepare a small sample dataset, clean it, perform exploratory data analysis, "
    "generate an EDA report, review it for quality, and confirm successful completion."
)

# Start conversation
result = user_proxy_agent.initiate_chat(
    manager,
    message=initial_task
)

In [ ]:
from IPython.display import Markdown, display

def display_chat_markdown(result):
    """
    Display the AutoGen multi-agent chat history in a clean, readable Markdown format.
    Works well in Colab or Jupyter environments.
    """
    if not result or not hasattr(result, "chat_history"):
        display(Markdown("⚠️ **No valid chat history found.**"))
        return

    md_output = "## 🤖 Multi-Agent Conversation Log\n\n"

    role_icons = {
        "userproxy": "🧑‍💻",
        "user_proxy": "🧑‍💻",
        "assistant": "🤖",
        "datapreparationagent": "🧹",
        "data_preparation_agent": "🧹",
        "edaagent": "📊",
        "eda_agent": "📊",
        "reportgeneratoragent": "📝",
        "report_generator_agent": "📝",
        "criticagent": "🕵️",
        "critic_agent": "🕵️",
        "executoragent": "⚙️",
        "executor_agent": "⚙️",
        "adminagent": "🧭",
        "admin_agent": "🧭"
    }

    # Build markdown string
    for i, msg in enumerate(result.chat_history, 1):
        role = msg.get("role", "unknown").lower()
        content = str(msg.get("content", "")).strip()

        # Choose emoji
        emoji = role_icons.get(role, "💬")

        # Escape potential markdown conflicts
        safe_content = content.replace("**", "\\*\\*")

        md_output += f"### {emoji} {role.capitalize()}\n\n"
        md_output += f"{safe_content}\n\n---\n"

        # Optional stop at TERMINATE
        if "TERMINATE" in content.upper():
            md_output += "✅ **Process terminated successfully.**\n"
            break

    # ✅ Ensure Markdown gets a string, not an object
    display(Markdown(str(md_output)))


##Groupchat output

In [ ]:
display_chat_markdown(result)

##

##Data Visualization

In [ ]:
import re

def extract_code_blocks(chat_result):
    """
    Extract Python code blocks from the chat history.
    Returns a list of code snippets.
    """
    code_blocks = []
    pattern = r"```python(.*?)```"

    for msg in chat_result.chat_history:
        content = msg.get("content", "")
        matches = re.findall(pattern, content, re.DOTALL)
        code_blocks.extend([m.strip() for m in matches])

    return code_blocks

In [ ]:
code_snippets = extract_code_blocks(result)

for i, code in enumerate(code_snippets):
    print(f"\n📊 Executing code block {i+1}...\n{'='*60}\n")
    try:
        exec(code, globals())  # Executes in your notebook
    except Exception as e:
        print(f"⚠️ Error in block {i+1}: {e}")

##References

ChatGPT
